# Neuromodulatory tuning of attentional sampling

This notebook continues the active-vision tutorial by asking a more specific question:

> **How should an event-driven attentional system balance exploration and exploitation?**

The sensory input is kept fixed: the same drift-generated RGB sequence is converted into the same DVS event stream, and the same proto-object saliency computation is used throughout. We then vary only the **fixation-selection policy**.

We proceed with two experiments:

1. **Softmax versus Argmax.** We compare stochastic exploratory sampling with deterministic winner-take-all selection.
   - **Softmax** is a stochastic fixation policy. Its inverse-temperature parameter $\beta$ controls how strongly saliency biases the selection of the next fixation.
   - **Argmax** is a deterministic winner-take-all control. It always selects the maximally salient location and has no $\beta$ parameter.

2. **Softmax gain sweep.** We keep the policy stochastic and vary the inverse-temperature parameter $\beta$ to test whether the computational system exhibits the expected exploration--exploitation trade-off observed in biological systems, and whether an intermediate gain can outperform both highly diffuse and highly selective sampling.

For every object and stochastic Softmax condition, we use **five seeds**. Seed-level runs constitute repeated measurements of the same visual stimulus and are therefore averaged within each object before statistical inference is performed across objects. We quantify performance using the following readouts:

- **explored object area**: the fraction of event-defined object cells visited by the fixation sequence;
- **normalised fixation entropy**: the extent to which fixations are distributed across those object cells.

## 1. Setup

We begin by importing the rendering, event-generation, attention, and analysis utilities used throughout the tutorial.

No experimental manipulation is performed at this stage. The goal is simply to establish a common pipeline so that all subsequent conditions differ only in the variable that we explicitly change: the **fixation-selection policy**.


In [ ]:
import sys
from pathlib import Path

repo = Path.cwd()
sys.path.insert(0, str(repo / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, HTML

from scene.render_offline import (
    render_camera_motion_sequence,
    frames_to_gif,
    frames_to_events_npy_and_gif,
)

from attention.attention import run_attention

from analysis_helpers import (
    display_gif_grid,
    compute_attention_exploration_metrics,
    plot_attention_exploration,
    plot_rgb_dvs_snapshot_grid,
    plot_attentional_gain_concept,
)

try:
    import bpy
    print("Blender Python available:", bpy.app.version_string)
except Exception as e:
    print("Warning: bpy could not be imported.")
    print(e)

## 1.1. Experimental configuration

We first define the objects and the parameters that will remain fixed throughout the experiments.

The central idea of this notebook is to manipulate the **attentional selection policy** while keeping the sensory input and saliency computation unchanged. This separation is important: if the RGB motion, DVS conversion, or proto-object computation changed between conditions, differences in exploration could not be attributed specifically to the fixation policy.

We therefore use:

- the same six objects;
- the same camera drift and rendering parameters;
- the same DVS sensor parameters;
- the same proto-object saliency computation;
- the same spatial definition of object cells.

Only the fixation-selection policy changes.

For stochastic Softmax conditions, each object is evaluated with **five random seeds**. Argmax is deterministic and therefore requires only one run per object.

> **Tutorial question:** Why is it important to keep the DVS stream fixed when comparing Softmax and Argmax?


In [ ]:
OBJECT_SPECS = [
    {
        "name": "airplane",
        "relative_path": "data/airplane_010.blend",
    },
    {
        "name": "apple",
        "relative_path": "data/apple_020.blend",
    },
    {
        "name": "hammer",
        "relative_path": "data/hammer_023.blend",
    },
    {
        "name": "bus",
        "relative_path": "data/bus_012.blend",
    },
    {
        "name": "laptop",
        "relative_path": "data/laptop_036.blend",
    },
    {
        "name": "cat",
        "relative_path": "data/cat_064.blend",
    },
]

OUTPUT_ROOT = repo / "data" / "renders" / "nm_beta_sampling"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

RESOLUTION = 256
FPS = 1000
NUM_FRAMES = 500
WINDOW_PERIOD_MS = 10.0

RENDER_PARAMS = dict(
    resolution=RESOLUTION,
    samples=1,
    use_gpu=True,
    object_target_size=0.45,
    object_azimuth_deg=315.0,
    object_elevation_deg=30.0,
    camera_position=(0.0, -0.25, 2.0),
    camera_target=(0.0, 0.0, 0.0),
    focal_length=50.0,
    sensor_width_mm=32.0,
    light_location=(0.0, 0.0, 5.0),
    light_size=10.0,
    light_strength=50.0,
    drift_sigma_deg=(0.10, 0.09),
)

EVENT_PARAMS = dict(
    fps=FPS,
    th_pos=0.15,
    th_neg=0.15,
    th_noise=0.05,
    lat=500,
    tau=300,
    jit=100,
    bgnp=0.001,
    bgnn=0.001,
    ref=40,
    skip_frames=0,
    gif_fps=20,
    gif_window_us=1000,
    loop=0,
)

ATTENTION_PARAMS_BASE = {
    "saliency_backend": "lif_vm",
    "num_pyr": 4,
    "lif_thetas": np.arange(0.0, 2.0 * np.pi, np.pi / 4),
    "lif_tau_mem": 0.3,
    "lif_size_krn": 16,
    "lif_rho": 0.1,
    "lif_r0": 14,
    "lif_thick": 3.0,
    "lif_offset": (0, 0),
    "lif_filter_resize_perc": 1.0,
    "lif_stride": 1,
    "lif_out_ch": 1,
    "lif_device": "auto",
    "lif_stateful": True,
}

# Experiment 1
SOFTMAX_COMPARE_BETA = 0.5

# Experiment 2
BETA_SWEEP = [0.5, 1.0, 2.5, 5.0, 10.0, 25.0, 50.0]

# Five stochastic repetitions
SEEDS = [0, 1, 2, 3, 4]

GRID_PER = 0.1
NOISE_THRESH = 100
EXCLUDE_INITIAL_FIXATION = True

# Consistent figure colours
COLORS = {
    "exploration": "navy",
    "wta": "crimson",
    "paired": "slategray",
}

## 2. Generate the common visual input

Before manipulating attention, we first construct the sensory input that will be shared by all attentional conditions.

For each object, we:

1. render a short RGB sequence containing small camera drift;
2. convert the RGB sequence into DVS events;
3. store the resulting event stream for reuse by all subsequent experiments.

The drift is useful because a static object would generate few or no events in an ideal event camera. Small image motion produces temporal contrast at object boundaries and internal features, making the object visible to the event-driven attention system.

Crucially, **we generate this stream only once per object**. Softmax and Argmax later operate on exactly the same `events.npy` file.

The next two code cells provide two possible workflows:

- `prepare_single_object_stream(...)` generates the RGB and DVS data from scratch;
- `load_existing_object_stream(...)` reloads previously generated outputs without running the renderer or DVS simulator again.

When reproducing the experiment from scratch, use the first route. When returning to an existing experiment, the second route avoids unnecessary recomputation.


In [ ]:
def prepare_single_object_stream(spec, seed=0):
    object_name = spec["name"]
    object_path = repo / spec["relative_path"]

    if not object_path.exists():
        raise FileNotFoundError(
            f"Object not found: {object_path}"
        )

    obj_root = (
        OUTPUT_ROOT
        / object_name
    )

    sequence_dir = (
        obj_root
        / "motion_sequence"
    )

    events_dir = (
        sequence_dir
        / "events"
    )

    seq = render_camera_motion_sequence(
        object_path=object_path,
        output_dir=sequence_dir,
        num_frames=NUM_FRAMES,
        fps=FPS,
        seed=seed,
        **RENDER_PARAMS,
    )

    rgb_gif_path = (
        sequence_dir
        / "rgb_motion.gif"
    )

    frames_to_gif(
        frames_dir=seq["frames_dir"],
        output_gif=rgb_gif_path,
        fps=30,
        loop=0,
    )

    ev = frames_to_events_npy_and_gif(
        frames_dir=seq["frames_dir"],
        output_dir=events_dir,
        **EVENT_PARAMS,
    )

    return {
        "name": object_name,
        "object_path": object_path,
        "sequence_dir": sequence_dir,
        "frames_dir": Path(
            seq["frames_dir"]
        ),
        "rgb_gif_path": rgb_gif_path,
        "events_dir": events_dir,
        "events_npy": Path(
            ev["npy_path"]
        ),
        "events_gif_path": Path(
            ev["gif_path"]
        ),
        "events_dat": Path(
            ev["dat_path"]
        ),
    }


object_runs = [
    prepare_single_object_stream(
        spec,
        seed=0,
    )
    for spec in OBJECT_SPECS
]

print(
    f"Prepared "
    f"{len(object_runs)} "
    f"object streams."
)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def load_existing_object_stream(spec):
    object_name = spec["name"]
    object_path = repo / spec["relative_path"]

    obj_root = OUTPUT_ROOT / object_name
    sequence_dir = obj_root / "motion_sequence"

    frames_dir = sequence_dir / "frames"
    rgb_gif_path = sequence_dir / "rgb_motion.gif"

    events_dir = sequence_dir / "events"
    events_npy = events_dir / "events.npy"
    events_gif_path = events_dir / "events.gif"
    events_dat = events_dir / "events.dat"

    required = [
        frames_dir,
        rgb_gif_path,
        events_npy,
        events_gif_path,
    ]

    missing = [
        p
        for p in required
        if not p.exists()
    ]

    if missing:
        raise FileNotFoundError(
            f"Missing outputs for {object_name}:\n"
            + "\n".join(
                f"  {p}"
                for p in missing
            )
        )

    return {
        "name": object_name,
        "object_path": object_path,
        "sequence_dir": sequence_dir,
        "frames_dir": frames_dir,
        "rgb_gif_path": rgb_gif_path,
        "events_dir": events_dir,
        "events_npy": events_npy,
        "events_gif_path": events_gif_path,
        "events_dat": events_dat,
    }


object_runs = [
    load_existing_object_stream(spec)
    for spec in OBJECT_SPECS
]

objects_order = [
    spec["name"]
    for spec in OBJECT_SPECS
]

print(
    f"Loaded {len(object_runs)} "
    f"existing object streams."
)

for run in object_runs:
    print(
        f"{run['name']:10s} | "
        f"RGB={run['rgb_gif_path'].exists()} | "
        f"DVS={run['events_npy'].exists()}"
    )

In [ ]:
rgb_paths = [run["rgb_gif_path"] for run in object_runs]
rgb_titles = [run["name"] for run in object_runs]

display_gif_grid(
    rgb_paths,
    rgb_titles,
    ncols=3,
    cell_width=320,
)

In [ ]:
dvs_paths = [run["events_gif_path"] for run in object_runs]
dvs_titles = [run["name"] for run in object_runs]

display_gif_grid(
    dvs_paths,
    dvs_titles,
    ncols=3,
    cell_width=320,
)

In [ ]:
out = plot_rgb_dvs_snapshot_grid(
    object_runs=object_runs,
    rgb_frame_idx=20,
    resolution=(RESOLUTION, RESOLUTION),
    fps=FPS,
    dvs_window_us=EVENT_PARAMS["gif_window_us"],
    figures_dir="figures",
    save_png=True,
    save_pdf=True,
    save_svg=True,
    show=True,
)

### Inspecting the sensory input

Before introducing attention, it is worth checking what information is actually available to the model.

The RGB and DVS panels represent the same physical scene, but they encode very different information. The RGB image contains static appearance, whereas the DVS representation contains only temporal changes produced by the imposed drift.

This means that the attention mechanism cannot simply recover an RGB segmentation of the object. Instead, it operates on the spatial structure produced by event activity.

> **Check your understanding**
>
> - Which parts of each object generate the strongest event activity?
> - Are object boundaries more visible than homogeneous interior regions?
> - Do all six objects produce similarly dense event representations?
> - Why might differences in event density matter when defining an event-based proxy for object area?


## 3. Proto-object saliency and fixation-selection policy

As in the previous tutorial, the attention module processes the event stream in short temporal windows. For each window, events are accumulated into a 2D event frame and transformed into a **proto-object saliency map** using a multiscale centre-surround computation. The resulting saliency map $S(x,y)$ highlights spatially coherent regions with strong local event structure, providing a bottom-up estimate of where informative object features are likely to be located.

Importantly, **saliency computation and fixation selection are separate operations**. Throughout this notebook, the event input and saliency computation are kept fixed. We modify only the policy used to convert the saliency map into the next fixation location.

- **Softmax sampling.** Under the stochastic policy, the next fixation is sampled from a probability distribution over the saliency map: $P(i \mid S,\beta) = \frac{\exp(\beta S_i)}{\sum_j \exp(\beta S_j)}$, where $S_i$ is the normalised saliency at location $i$ and $\beta$ is an inverse-temperature parameter controlling how strongly saliency biases fixation selection. In other words, $\beta$ regulates the balance between broad exploration and focused exploitation.

- **Argmax selection.** As a deterministic winner-take-all control, the next fixation is selected directly from the maximally salient location: $i^\star = \arg\max_i S_i$. Argmax therefore uses the same event stream and the same proto-object saliency map as Softmax, but removes stochastic exploration entirely, corresponding to the purely exploitative limit. Consistently, as $\beta \rightarrow \infty$, the Softmax distribution becomes increasingly concentrated at the maximally salient location and approaches Argmax selection.

In [ ]:
out = plot_attentional_gain_concept(
    figures_dir="figures",
    save_png=True,
    save_pdf=True,
    save_svg=True,
    show=True,
)

### From gain to a testable prediction

The parameter $\beta$ does not change the saliency map itself. It changes how strongly differences in saliency influence the probability of selecting the next fixation.

This leads to three qualitatively different regimes:

- **low $\beta$ — exploratory:** the probability distribution is relatively diffuse, allowing attention to sample locations that are not necessarily the current saliency maximum;
- **intermediate $\beta$ — selective exploration:** salient regions are favoured, while stochasticity still allows transitions between different regions;
- **very high selectivity / Argmax — exploitative:** attention repeatedly favours the strongest available location.

A simple hypothesis would be that decreasing $\beta$ always improves exploration. However, this need not be true. If the distribution becomes too diffuse, saliency provides progressively less guidance and a limited fixation budget may be used less efficiently.

This motivates the two experiments below.

> **Prediction:** If useful exploration requires both stochasticity and saliency guidance, the best object coverage should emerge at an intermediate degree of attentional selectivity rather than at either extreme.


## 4. Experiments

The conceptual figure above illustrates how Softmax gain controls the exploration--exploitation balance. We now quantify this behaviour on real DVS streams using two complementary readouts:

1. **Explored object area** — a proxy for how much of the event-defined object is visited by attention. To obtain this measure, we divide the visual field into a regular spatial grid and use the accumulated DVS activity to determine which cells are likely to belong to the object. Specifically, a grid cell is labelled as an **object cell** when the total number of events falling within that cell exceeds a predefined threshold. Thus, the object is defined directly from its event activity rather than from a ground-truth segmentation mask. A cell is then considered **visited** if at least one fixation falls within it.

   The explored object area is therefore computed as

   $$
   A_{\mathrm{explored}}
   =
   \frac{N_{\mathrm{visited\ object\ cells}}}
   {N_{\mathrm{object\ cells}}}.
   $$

   This quantity should be interpreted as an **event-based proxy for spatial object coverage**, rather than as the fraction of the object's true geometric area that has been observed. A value close to $1$ indicates that attention has visited most regions exhibiting substantial object-related event activity, whereas a low value indicates that fixations remain concentrated within a small part of that event-defined region. In the current implementation, the grid-cell size is set relative to the image dimensions and cells containing at least `noise_thresh` accumulated events are treated as object cells.

2. **Normalised fixation entropy** — a complementary measure of how evenly attention is distributed across the detected object cells. Whereas explored object area asks **how many different object regions are reached**, fixation entropy asks **how evenly fixations are allocated among them**. For each object cell $k$, we compute the fraction of fixations assigned to that cell, $p_k$. The fixation entropy is then $H = -\sum_k p_k \log p_k$. A low entropy indicates that fixations are concentrated within a small number of object cells, whereas a high entropy indicates that attention is distributed more broadly across the object. Because the maximum possible entropy depends on the number of detected object cells $K$, we report the normalised entropy $H_{\mathrm{norm}} = \frac{H}{\log K}$. This maps the measure onto a comparable scale across objects with different numbers of event-defined cells: values near $0$ indicate strongly concentrated sampling, while values near $1$ indicate a nearly uniform distribution of fixations across the available object cells.

Together, these two measures capture related but distinct aspects of attentional exploration. **Explored object area measures spatial coverage**, whereas **normalised fixation entropy measures the diversity of sampling within that space**.



### What counts as one statistical observation?

There are two levels of repetition in these experiments, and they should not be confused.

For Softmax, changing the random seed generates different stochastic fixation trajectories for the **same object and the same event stream**. These runs therefore quantify variability in the sampling policy, but they are not independent objects.

We consequently use the following hierarchy:

$$
\text{five Softmax seeds}
\;\longrightarrow\;
\text{mean within each object}
\;\longrightarrow\;
\text{comparison across objects}.
$$

The six objects are the units used for the paired statistical comparisons.

This avoids treating multiple stochastic realizations of the same stimulus as independent samples — a form of pseudoreplication.


In [ ]:
def run_attention_once(
    events_npy,
    out_dir,
    mode="softmax",
    beta=None,
    seed=0,
):
    params = dict(
        ATTENTION_PARAMS_BASE
    )

    params["mode"] = mode
    params["seed"] = int(seed)

    if mode == "softmax":
        if beta is None:
            raise ValueError(
                "Softmax mode requires beta."
            )

        params["beta"] = float(beta)

    elif mode == "argmax":
        params.pop(
            "beta",
            None,
        )

    else:
        raise ValueError(
            f"Unknown attention mode: {mode}"
        )

    out_dir = Path(out_dir)

    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    return run_attention(
        events_npy=events_npy,
        output_dir=out_dir,
        resolution=(
            RESOLUTION,
            RESOLUTION,
        ),
        window_period_ms=WINDOW_PERIOD_MS,
        max_windows=None,
        sigma=None,
        use_polarity=False,
        clear_existing=True,
        attention_params=params,

        # No plotting during experimental sweeps
        plot=False,
        plot_gif_path=None,
        plot_fps=10,
        plot_loop=0,
    )

## 4.1. Experiment 1 — Does stochastic sampling improve exploration?

The first experiment isolates the contribution of **stochastic exploration**.

We compare:

- **Softmax Exploration**, using $\beta=0.5$;
- **Argmax Exploitation**, which always selects the maximally salient location.

Everything upstream of fixation selection is identical between the two conditions. The same object generates the same RGB motion, the same DVS stream, and the same proto-object saliency computation.

For Softmax, five stochastic trajectories are generated for each object and averaged to obtain one Softmax value per object. Argmax is deterministic, so one trajectory is sufficient.

The comparison therefore has a paired structure:

$$
\mathrm{object}_i:
\qquad
\mathrm{Softmax}_i
\;\longleftrightarrow\;
\mathrm{Argmax}_i.
$$

Our hypothesis is that removing stochastic exploration entirely will cause attention to revisit a restricted set of highly salient regions, reducing both **object coverage** and **fixation diversity**.


In [ ]:
softmax_argmax_rows = []

for run in object_runs:

    object_name = run["name"]
    events_npy = run["events_npy"]

    # ============================================================
    # Softmax: 5 stochastic seeds
    # ============================================================
    for seed in SEEDS:

        print(
            f"{object_name} | "
            f"softmax | "
            f"beta={SOFTMAX_COMPARE_BETA} | "
            f"seed={seed}"
        )

        out_dir = (
            run["sequence_dir"]
            / "attention_experiments"
            / "softmax_vs_argmax"
            / f"softmax_beta_{SOFTMAX_COMPARE_BETA}"
            / f"seed_{seed}"
        )

        att = run_attention_once(
            events_npy=events_npy,
            out_dir=out_dir,
            mode="softmax",
            beta=SOFTMAX_COMPARE_BETA,
            seed=seed,
        )

        metrics = (
            compute_attention_exploration_metrics(
                events_npy=events_npy,
                saccades_path=att["saccades_path"],
                resolution=(
                    RESOLUTION,
                    RESOLUTION,
                ),
                per=GRID_PER,
                noise_thresh=NOISE_THRESH,
                exclude_initial_fixation=(
                    EXCLUDE_INITIAL_FIXATION
                ),
            )
        )

        softmax_argmax_rows.append({
            "object": object_name,
            "mode": "softmax",
            "beta": SOFTMAX_COMPARE_BETA,
            "seed": seed,
            **metrics,
        })

    # ============================================================
    # Argmax: deterministic control
    # ============================================================
    print(
        f"{object_name} | argmax"
    )

    out_dir = (
        run["sequence_dir"]
        / "attention_experiments"
        / "softmax_vs_argmax"
        / "argmax"
    )

    att = run_attention_once(
        events_npy=events_npy,
        out_dir=out_dir,
        mode="argmax",
        beta=None,
        seed=0,
    )

    metrics = (
        compute_attention_exploration_metrics(
            events_npy=events_npy,
            saccades_path=att["saccades_path"],
            resolution=(
                RESOLUTION,
                RESOLUTION,
            ),
            per=GRID_PER,
            noise_thresh=NOISE_THRESH,
            exclude_initial_fixation=(
                EXCLUDE_INITIAL_FIXATION
            ),
        )
    )

    softmax_argmax_rows.append({
        "object": object_name,
        "mode": "argmax",
        "beta": np.nan,
        "seed": 0,
        **metrics,
    })


softmax_argmax_df = pd.DataFrame(
    softmax_argmax_rows
)

softmax_argmax_df.to_csv(
    OUTPUT_ROOT
    / "softmax_vs_argmax_metrics.csv",
    index=False,
)

softmax_argmax_df

In [ ]:
softmax_argmax_object_df = (
    softmax_argmax_df
    .groupby(
        [
            "object",
            "mode",
        ],
        as_index=False,
    )
    .agg(
        area_explored_coeff=(
            "area_explored_coeff",
            "mean",
        ),
        fixation_entropy_norm=(
            "fixation_entropy_norm",
            "mean",
        ),
        fixation_entropy=(
            "fixation_entropy",
            "mean",
        ),
        num_fixations=(
            "num_fixations",
            "mean",
        ),
        num_fixations_on_object=(
            "num_fixations_on_object",
            "mean",
        ),
    )
)

softmax_argmax_object_df

### Statistical test: exact paired sign-flip permutation test

To test whether Softmax and Argmax differ systematically across objects, we use an **exact two-sided paired sign-flip permutation test**.

The test operates on the six object-level paired differences:

$$
d_i
=
x_i^{\mathrm{Softmax}}
-
x_i^{\mathrm{Argmax}},
$$

where $i$ indexes objects.

Under the null hypothesis that there is no systematic difference between the two conditions, the direction of each paired difference is exchangeable: a difference of $+d_i$ is considered equally compatible with the null as $-d_i$.

The observed statistic is the absolute mean paired difference,

$$
T_{\mathrm{obs}}
=
\left|
\frac{1}{N}
\sum_i d_i
\right|.
$$

Because there are only $N=6$ objects, we do not need to approximate the null distribution. We can enumerate **all $2^6=64$ possible sign assignments** of the paired differences and recompute the statistic for every permutation.

The exact two-sided p-value is the proportion of those permutations producing a statistic at least as large as the observed one.

#### Why use this test?

This test is particularly appropriate here because:

- the observations are naturally **paired by object**;
- the number of objects is small;
- it does not rely on a large-sample Gaussian approximation;
- the complete permutation space is small enough to enumerate exactly.

The five Softmax seeds are averaged **before** this test. They characterize stochastic variability within an object but are not treated as five independent samples.

#### What do the stars mean?

The plots use the conventional shorthand:

$$
\begin{aligned}
\mathrm{ns} &: p \geq 0.05,\\
* &: p < 0.05,\\
** &: p < 0.01,\\
*** &: p < 0.001,\\
**** &: p < 0.0001.
\end{aligned}
$$

The exact numerical p-values are also printed by the plotting function and should be preferred when reporting the result.

> **Small-sample consequence:** with six paired objects, the exact two-sided sign-flip test has a discrete p-value resolution. If all six paired differences point maximally consistently in one direction, the smallest attainable two-sided p-value is
>
> $$
> \frac{2}{2^6}=0.03125.
> $$
>
> Therefore, with the present sample size, this exact test can produce a significant `*`, but cannot reach `**` or `***`. This is a property of the exact test and the number of independent objects, not of the plotting code.


In [ ]:
from analysis_helpers import (
    plot_softmax_argmax_overview_1x4,
)

objects_order = [
    spec["name"]
    for spec in OBJECT_SPECS
]

out = plot_softmax_argmax_overview_1x4(
    softmax_argmax_df=softmax_argmax_df,
    softmax_argmax_object_df=softmax_argmax_object_df,
    objects_order=objects_order,
    softmax_beta=0.5,
    figures_dir="figures",
    save_png=True,
    save_pdf=True,
    save_svg=True,
    show=True,
)

### Interpreting Experiment 1

The by-object plots retain the identity of each stimulus and show how the effect varies across objects. The paired plots summarize the same comparison at the object level.

The important question is not simply whether two averages differ. Because the experiment is paired, we should also inspect whether **the same direction of change is reproduced across objects**.

> **Questions**
>
> 1. Does Softmax increase explored object area for most individual objects?
> 2. Does the same pattern appear for normalised fixation entropy?
> 3. Are the effects driven by one unusual object, or are the paired trajectories reasonably consistent?
> 4. What does lower entropy under Argmax imply about repeated fixation of the same object regions?
> 5. Does this experiment establish that lower $\beta$ is always better?
>
> The final question motivates the next experiment. Softmax may outperform deterministic Argmax while still exhibiting a non-trivial trade-off **within the stochastic regime itself**.


## 4.2. Experiment 2 — How much exploration is useful?

Experiment 1 compares stochastic exploration with deterministic exploitation. We now keep the policy **strictly Softmax** and ask a different question:

> **Does exploration continue to improve as the Softmax distribution becomes increasingly diffuse?**

We sweep

$$
\beta
\in
\{0.5,\;1,\;5,\;10,\;25,\;50\}
$$

while keeping the event stream and saliency computation unchanged.

The two extremes have different potential failure modes:

- at **high $\beta$**, sampling becomes strongly concentrated around the most salient locations, potentially causing repeated fixation of a restricted region;
- at **very low $\beta$**, sampling becomes increasingly diffuse, weakening the guidance provided by proto-object saliency.

An intermediate $\beta$ could therefore provide a useful compromise: sufficiently stochastic to visit different object regions, but sufficiently selective to remain guided by event-defined object structure.

For every object and every $\beta$, five stochastic seeds are generated and subsequently averaged within object.


In [ ]:
beta_rows = []

for run in object_runs:

    object_name = run["name"]
    events_npy = run["events_npy"]

    for beta in BETA_SWEEP:

        beta_token = str(
            beta
        ).replace(
            ".",
            "p",
        )

        for seed in SEEDS:

            print(
                f"{object_name} | "
                f"beta={beta} | "
                f"seed={seed}"
            )

            out_dir = (
                run["sequence_dir"]
                / "attention_experiments"
                / "beta_sweep"
                / f"beta_{beta_token}"
                / f"seed_{seed}"
            )

            att = run_attention_once(
                events_npy=events_npy,
                out_dir=out_dir,
                mode="softmax",
                beta=beta,
                seed=seed,
            )

            metrics = (
                compute_attention_exploration_metrics(
                    events_npy=events_npy,
                    saccades_path=att[
                        "saccades_path"
                    ],
                    resolution=(
                        RESOLUTION,
                        RESOLUTION,
                    ),
                    per=GRID_PER,
                    noise_thresh=NOISE_THRESH,
                    exclude_initial_fixation=(
                        EXCLUDE_INITIAL_FIXATION
                    ),
                )
            )

            beta_rows.append({
                "object": object_name,
                "beta": beta,
                "seed": seed,
                **metrics,
            })


beta_df = pd.DataFrame(
    beta_rows
)

beta_df.to_csv(
    OUTPUT_ROOT
    / "beta_sweep_metrics.csv",
    index=False,
)

beta_df

In [ ]:
beta_object_df = (
    beta_df
    .groupby(
        [
            "object",
            "beta",
        ],
        as_index=False,
    )
    .agg(
        area_explored_coeff=(
            "area_explored_coeff",
            "mean",
        ),
        fixation_entropy_norm=(
            "fixation_entropy_norm",
            "mean",
        ),
        fixation_entropy=(
            "fixation_entropy",
            "mean",
        ),
        num_fixations=(
            "num_fixations",
            "mean",
        ),
        num_fixations_on_object=(
            "num_fixations_on_object",
            "mean",
        ),
    )
)

beta_object_df.to_csv(
    OUTPUT_ROOT
    / "beta_sweep_object_means.csv",
    index=False,
)

beta_object_df

### From stochastic runs to object trajectories

The raw sweep contains five stochastic realizations for every object–$\beta$ combination.

Before plotting the systematic effect of gain, we average those repetitions within each object:

$$
\bar{y}_{i,\beta}
=
\frac{1}{5}
\sum_{s=1}^{5}
y_{i,\beta,s}.
$$

Each coloured trajectory in the following figure therefore corresponds to **one object followed across the complete gain sweep**.

This repeated-measures view is useful because it separates two sources of variation:

- differences between objects;
- systematic changes produced by $\beta$ within the same object.

The dashed curve provides a descriptive summary of the overall shape of the sweep. It should not be interpreted as replacing the object-level paired statistical comparisons performed below.


In [ ]:
from analysis_helpers import plot_beta_sweep_by_object

out = plot_beta_sweep_by_object(
    beta_object_df=beta_object_df,
    objects_order=objects_order,
    figures_dir="figures",
    save_png=True,
    save_pdf=True,
    save_svg=True,
    show=True,
)

### Reducing the sweep to three functional regimes

The full sweep allows us to visualize how sampling changes continuously with attentional gain. We can now summarize the main functional regimes using three representative conditions:

- **Softmax Exploration:** $\beta=0.5$;
- **Softmax Balance:** $\beta=5$;
- **Argmax Exploitation:** deterministic winner-take-all selection.

This comparison is not intended to imply that $\beta=5$ is a universally optimal gain. Rather, among the values tested here, it represents the intermediate regime associated with high spatial coverage and fixation diversity.

We therefore ask two targeted questions:

1. Does the intermediate Softmax regime outperform the highly exploratory Softmax regime?
2. Does it outperform deterministic exploitation?

For each metric, these questions are tested using the same **exact paired sign-flip test** introduced in Experiment 1:

$$
\text{Exploration}
\leftrightarrow
\text{Balance}
$$

and

$$
\text{Balance}
\leftrightarrow
\text{Exploitation}.
$$

Again, the statistical unit is the object, not the individual stochastic seed.


In [ ]:
from analysis_helpers import (
    plot_exploration_balance_exploitation_paired,
)

objects_order = [
    spec["name"]
    for spec in OBJECT_SPECS
]

out = plot_exploration_balance_exploitation_paired(
    beta_object_df=beta_object_df,
    softmax_argmax_object_df=softmax_argmax_object_df,

    exploration_beta=0.5,
    balance_beta=5.0,

    objects_order=objects_order,
    figures_dir="figures",

    save_png=True,
    save_pdf=True,
    save_svg=True,
    show=True,
)

### Interpreting Experiment 2

The gain sweep tests a stronger hypothesis than the initial Softmax–Argmax comparison.

If exploration were beneficial without limit, the lowest $\beta$ should produce the highest coverage and fixation entropy. Conversely, if selectivity alone were sufficient, performance should continue to improve toward Argmax.

Instead, an intermediate peak would indicate that effective active sampling requires a balance between the two.

> **Questions**
>
> 1. Do explored object area and fixation entropy show the same qualitative dependence on $\beta$?
> 2. Is the intermediate regime consistently better across objects, or does the preferred gain vary substantially between stimuli?
> 3. Why might very low $\beta$ reduce object coverage even though it increases stochasticity?
> 4. Why does Argmax tend to reduce fixation entropy?
> 5. What additional measurement would be required to demonstrate directly that very diffuse sampling increases **off-object** fixations?
> 6. Would the same intermediate regime be expected if the fixation budget, object size, or event threshold changed?
>
> The present experiment therefore supports a computational interpretation in which **attentional gain regulates selective exploration**: too much selectivity restricts sampling, whereas too little selectivity weakens the guidance provided by saliency.
>
> This should be interpreted as a bio-inspired computational principle rather than as a direct experimental validation of a particular neuromodulatory mechanism.


### Neuromodulatory perspective

The computational mechanism explored in this tutorial also provides a useful point of contact with biological neuromodulation.

Neuromodulatory systems do not simply switch neural processing on or off. Instead, they can reconfigure how strongly neural populations respond to incoming information and how selectively that information influences ongoing behaviour. In the context of attention, neuromodulators such as norepinephrine, acetylcholine, and dopamine have been associated with changes in neuronal response gain, excitation–inhibition balance, population correlations, network stability, and the allocation of processing resources to behaviourally relevant information (Thiele & Bellgrove, 2018).

A particularly relevant example is the **adaptive gain theory** of the locus coeruleus–norepinephrine (LC–NE) system proposed by Aston-Jones and Cohen (2005). In that framework, different modes of LC activity are associated with different behavioural regimes: phasic LC responses support engagement with and exploitation of currently useful behaviour, whereas elevated tonic LC activity is associated with disengagement and increased exploration of alternatives.

The mechanism studied here is clearly much simpler, but it captures a related computational principle. By changing a single gain parameter, $\beta$, the same sensory representation can support qualitatively different sampling regimes:

$$
\text{diffuse exploration}
\;\longleftrightarrow\;
\text{selective exploration}
\;\longleftrightarrow\;
\text{deterministic exploitation}.
$$

Importantly, the intermediate regime is particularly informative. Our results suggest that useful exploration does not require maximal randomness. Instead, effective sampling emerges when stochasticity is sufficient to escape repeated selection of the same salient locations while the saliency representation remains strong enough to guide attention toward informative regions.

This resembles a broader principle attributed to neuromodulatory systems: **adaptive behaviour may depend not on maximizing either exploration or exploitation, but on dynamically regulating the operating regime of a neural circuit according to current behavioural demands.**

However, this analogy should not be interpreted as a direct biological model of LC–NE or any other neuromodulatory system. In particular, $\beta$ is neither a model of norepinephrine concentration nor a direct representation of tonic or phasic LC firing. Here it is a computational control variable acting on the stochasticity and selectivity of attentional sampling. The correspondence is therefore **functional and bio-inspired**: biological neuromodulators motivate the idea that a common circuit can be rapidly reconfigured between different behavioural regimes without changing its underlying sensory representation.

> **Take-home question:** If attentional gain were itself controlled dynamically rather than fixed, what internal signal could tell the system when to become more exploratory or more exploitative?

## 5. What did we learn?

This tutorial separated two components that are often conflated in active vision:

1. **where potentially relevant structure is represented**, through the proto-object saliency map;
2. **how that representation is sampled**, through the fixation-selection policy.

The first experiment showed how stochastic Softmax sampling can be compared directly with deterministic winner-take-all selection while holding sensory processing fixed.

The second experiment showed why the amount of stochasticity itself matters. Within Softmax, attentional gain controls a continuum from diffuse exploration to highly selective sampling, and the resulting exploration behaviour can be quantified through both spatial coverage and fixation diversity.

Together, the experiments illustrate a general principle:

> **Adaptive attention is not simply a choice between exploration and exploitation; it can be understood as the regulation of how strongly current sensory evidence constrains future sampling.**


## Optional: Reload previously computed results

The experiments above save their outputs to CSV files. When returning to the notebook later, the following cell can reload those results without rerunning rendering, DVS conversion, or attention sampling.

Use this shortcut only after the experiment CSV files have already been generated.


In [ ]:
softmax_argmax_csv = (
    OUTPUT_ROOT
    / "softmax_vs_argmax_metrics.csv"
)

beta_sweep_csv = (
    OUTPUT_ROOT
    / "beta_sweep_metrics.csv"
)

beta_object_csv = (
    OUTPUT_ROOT
    / "beta_sweep_object_means.csv"
)


for path in [
    softmax_argmax_csv,
    beta_sweep_csv,
    beta_object_csv,
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing experiment file: {path}"
        )


# ------------------------------------------------------------
# Seed-level data
# ------------------------------------------------------------
softmax_argmax_df = pd.read_csv(
    softmax_argmax_csv
)

beta_df = pd.read_csv(
    beta_sweep_csv
)


# ------------------------------------------------------------
# Object-level beta means
# ------------------------------------------------------------
beta_object_df = pd.read_csv(
    beta_object_csv
)


print(
    "softmax_argmax_df:",
    softmax_argmax_df.shape,
)

print(
    "beta_df:",
    beta_df.shape,
)

print(
    "beta_object_df:",
    beta_object_df.shape,
)